# tsfresh feature generation and t-sne analysis on StressID and ExpData datasets

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from dataclasses import dataclass
from typing import Tuple, TypeAlias, ClassVar

from tsfresh import extract_features, select_features
from tsfresh.feature_extraction.settings import EfficientFCParameters, MinimalFCParameters, IndexBasedFCParameters
from tsfresh.utilities.dataframe_functions import impute

BASE_PATH = "../../.."
DATASET_SEPARATOR = ";"
DATASET = f"{BASE_PATH}/experiment-data"
DATASET_FILENAME = f"{DATASET}/flat_dataset.alltasks.csv.gz"
LABELS_SEPARATOR = ","
LABELS = f"{DATASET}/labels.csv"

EXTRACTED_FEATURES_FILENAME = f"{DATASET}/tsfresh_extfeat.csv"
SELECTED_FEATURES_FILENAME = f"{DATASET}/tsfresh_selfeat.csv"

BIN_LABELS = ["NoStress", "Stress"]
TER_LABELS = ["Relaxed", "Stress", "RealStress"]
QAD_LABELS = ["Relaxed", "Stress", "RealStress", "Amused"]

LABELS_CONF = {
    "b": {
        "col_name": "binary-stress",
        "enabled": True,
        "stratification": True,
        "classes": BIN_LABELS,
    },
    "t": {
        "col_name": "affect3-class",
        "enabled": True,
        "stratification": True,
        "classes": TER_LABELS,
    },
    "q": {
        "col_name": "affect4-class",
        "enabled": True,
        "stratification": True,
        "classes": QAD_LABELS,
    },
}

In [10]:
class FullData:
    def __init__(self):
        self.subject: list[int] = []
        self.task: list[str] = []
        self.timestamp: list[float] = []
        self.eda: list[float] = []
        self.ppg: list[float] = []
        self.accel_x: list[float] = []
        self.accel_y: list[float] = []
        self.accel_z: list[float] = []
        self.gyro_x: list[float] = []
        self.gyro_y: list[float] = []
        self.gyro_z: list[float] = []
        self.temp: list[float] = []
        self.pressure: list[float] = []
        self.b_classes: list[int] = []
        self.t_classes: list[int] = []
        self.q_classes: list[int] = []

    def to_dataframe(self) -> pd.DataFrame:
        return pd.DataFrame(
            {
                "Subject": self.subject,
                "Task": self.task,
                "Timestamp": self.timestamp,
                "EDA": self.eda,
                "PPG": self.ppg,
                "Accel_X": self.accel_x,
                "Accel_Y": self.accel_y,
                "Accel_Z": self.accel_z,
                "Gyro_X": self.gyro_x,
                "Gyro_Y": self.gyro_y,
                "Gyro_Z": self.gyro_z,
                "Temperature": self.temp,
                "Pressure": self.pressure,
                "Bin_Class": self.b_classes,
                "Ter_Class": self.t_classes,
                "Qad_Class": self.q_classes,
            }
        )

    def assert_lengths(self) -> None:
        differs = False
        lengths = {}
        expected_len = len(self.timestamp)
        # print(f"Asserting the length of {expected_len} items on the columns")
        for name, instance_attr in self.__dict__.items():
            lengths[name] = len(instance_attr)
            if len(instance_attr) != expected_len:
                differs = True
        if differs:
            AssertionError(f"Columns are not the same lenght: {lengths}")


### Build dataset from data files

In [11]:
# Creating labels object

labels_df = pd.read_csv(LABELS, sep=LABELS_SEPARATOR, header=0, index_col=0)
tasks: dict[str, pd.Series] = {}
for key, conf in LABELS_CONF.items():
    if conf["enabled"]:
        tasks[key] = labels_df[conf["col_name"]]

display(tasks["b"])
tasks_with_labels: list[str] = tasks["b"].index


subject/task
01-AmusementClip    0
01-Baseline         0
01-EmoReset         0
01-FormL            1
01-FormM            1
                   ..
21-Baseline         0
21-EmoReset         0
21-FormL            0
21-FormM            0
21-StressClip       1
Name: binary-stress, Length: 126, dtype: int64

In [16]:
# Creating dataset file, it may be skipped

# data will not be normalized or standarized
# using calibrated data (no raw data since Shimmer3's GSR sensor requires range scaling)
CREATE_DATASET_FILE = False
EXPECTED_NUM_FILES = 21
SAMPLING_RATE = 51.2
NEUROCLEAN = True
CREATE_TEST = False
TEST_N_ITERATIONS = 3

if CREATE_DATASET_FILE:
    import glob
    import neurokit2 as nk
    type FeatureDict = dict[str, np.ndarray]

    col_types = {
        "Timestamp": float,
        "Event": str,
        "ExtraEvent": str,
        "AccelLN_X": float,
        "AccelLN_Y": float,
        "AccelLN_Z": float,
        "Battery": float,
        "GSR_Range": int,
        "Skin_Conductance": float,
        "Skin_Resistance": float,
        "Gyro_X": float,
        "Gyro_Y": float,
        "Gyro_Z": float,
        "PPG": float,
        "Pressure": float,
        "Temperature": float,
        "AccelLN_X_Uncal": int,
        "AccelLN_Y_Uncal": int,
        "AccelLN_Z_Uncal": int,
        "Skin_Conductance_Uncal": int,
        "PPG_Uncal": int,
    }

    ####### LOAD DATA
    filelist = glob.glob(f"{DATASET}/*.Annotated.csv")
    filelist.sort()
    if len(filelist) != EXPECTED_NUM_FILES:
        raise ValueError(f"Expected {EXPECTED_NUM_FILES} files, found: {len(filelist)}")

    full_data = FullData()
    subject_to_int: dict[str, int] = {}
    subject_counter = 0

    def split_by_task(df: pd.DataFrame) -> list[tuple[str, pd.DataFrame]]:
        output: list[tuple[str, pd.DataFrame]] = []
        tasks = ["Baseline", "AmusementClip", "StressClip", "EmoReset", "FormL", "FormM", "Debriefing"]
        start_idx = end_idx = 0
        for task in tasks:
            if task == "FormL":
                if "FormLRead" in df["Event"].values:
                    start_idx = df.index.get_loc(df[df["Event"] == "FormLRead"].index[0])
                    end_idx = df.index.get_loc(df[df["Event"] == "L15"].index[-1])
                else:
                    start_idx = df.index.get_loc(df[df["Event"] == "FormL"].index[0])
                    end_idx = df.index.get_loc(df[df["Event"] == "FormL"].index[-1])
            elif task == "FormM":
                if "FormMRead" in df["Event"].values:
                    start_idx = df.index.get_loc(df[df["Event"] == "FormMRead"].index[0])
                    end_idx = df.index.get_loc(df[df["Event"] == "M15"].index[-1])
                else:
                    start_idx = df.index.get_loc(df[df["Event"] == "FormM"].index[0])
                    end_idx = df.index.get_loc(df[df["Event"] == "FormM"].index[-1])
            else:
                start_idx = df.index.get_loc(df[df["Event"] == task].index[0])
                end_idx = df.index.get_loc(df[df["Event"] == task].index[-1])
            output.append((task, df[start_idx:end_idx]))
        return output

    iteration_counter = 0
    for item in filelist:
        file: pd.DataFrame = pd.read_csv(
            item,
            delimiter=";",
            date_format=r"%Y-%m-%d %H:%M:%S.%f",
            parse_dates=["Datetime", "Timestamp"],
            index_col=["Datetime"],
            dtype=col_types,
        )
        filename = item.split("/")[-1]
        subject_id = filename.split("-")[1]
        if subject_id not in subject_to_int:
            subject_to_int[subject_id] = subject_counter
            subject_counter += 1

        for task, event_df in split_by_task(file):
            label_id = f"{subject_id}-{task}"
            if label_id not in tasks_with_labels:
                continue
            n_elements = event_df["Timestamp"].size
            ppg_signal = (
                nk.ppg_clean(np.array(event_df["PPG"]), sampling_rate=SAMPLING_RATE)
                if NEUROCLEAN
                else event_df["PPG"].to_list()
            )
            eda_signal = (
                nk.eda_clean(np.array(event_df["Skin_Conductance"]), sampling_rate=SAMPLING_RATE, method="neurokit")
                if NEUROCLEAN
                else event_df["Skin_Conductance"].to_list()
            )
            if ppg_signal.size != n_elements:
                raise Exception(f"Sizes differ {ppg_signal.size} vs {n_elements}")
            full_data.subject.extend(np.full(n_elements, subject_to_int[subject_id]))
            full_data.task.extend(np.full(n_elements, task))
            full_data.timestamp.extend(event_df["Timestamp"].to_list())
            full_data.ppg.extend(ppg_signal)
            full_data.eda.extend(eda_signal)
            full_data.accel_x.extend(event_df["AccelLN_X"].to_list())
            full_data.accel_y.extend(event_df["AccelLN_Y"].to_list())
            full_data.accel_z.extend(event_df["AccelLN_Z"].to_list())
            full_data.gyro_x.extend(event_df["Gyro_X"].to_list())
            full_data.gyro_y.extend(event_df["Gyro_Y"].to_list())
            full_data.gyro_z.extend(event_df["Gyro_Z"].to_list())
            full_data.temp.extend(event_df["Temperature"].to_list())
            full_data.pressure.extend(event_df["Pressure"].to_list())
            full_data.b_classes.extend(np.full(n_elements, tasks["b"][label_id]))
            full_data.t_classes.extend(np.full(n_elements, tasks["t"][label_id]))
            full_data.q_classes.extend(np.full(n_elements, tasks["q"][label_id]))
            full_data.assert_lengths()

        iteration_counter += 1
        if CREATE_TEST and iteration_counter >= TEST_N_ITERATIONS:
            break

    fulldata_df = full_data.to_dataframe()
    if not CREATE_TEST:
        fulldata_df.to_csv(DATASET_FILENAME, sep=DATASET_SEPARATOR, index=False, compression="gzip")
    display(fulldata_df)


In [17]:
DESIRED_COLUMNS = ["Subject", "Task", "Timestamp", "EDA", "PPG"]
DESIRED_LABELS = "Bin_Class"

In [18]:
fulldata_df = pd.read_csv(DATASET_FILENAME, sep=DATASET_SEPARATOR, compression="gzip")
X = fulldata_df[DESIRED_COLUMNS]
y = fulldata_df[DESIRED_LABELS]
X_groups = fulldata_df["Subject"]

In [21]:
display(X)
display(y)
display(X_groups)
samples_stats_df = X.groupby(["Subject", "Task"]).agg(samples=("Timestamp", "size"))
samples_stats_df["Duration(min)"] = samples_stats_df["samples"] / SAMPLING_RATE / 60.0

display(samples_stats_df)

,Subject,Task,Timestamp,EDA,PPG
0,0,Baseline,1.749609e+12,1.397531,-8.688205
1,0,Baseline,1.749609e+12,1.403573,-17.615588
2,0,Baseline,1.749609e+12,1.409202,-26.264472
3,0,Baseline,1.749609e+12,1.414017,-34.198909
4,0,Baseline,1.749609e+12,1.417698,-40.997525
...,...,...,...,...,...
1435016,20,FormM,1.752561e+12,6.794845,41.209776
1435017,20,FormM,1.752561e+12,6.801706,5.093412
1435018,20,FormM,1.752561e+12,6.808988,-24.224583
1435019,20,FormM,1.752561e+12,6.816556,-42.873762


0          0
1          0
2          0
3          0
4          0
          ..
1435016    0
1435017    0
1435018    0
1435019    0
1435020    0
Name: Bin_Class, Length: 1435021, dtype: int64

0           0
1           0
2           0
3           0
4           0
           ..
1435016    20
1435017    20
1435018    20
1435019    20
1435020    20
Name: Subject, Length: 1435021, dtype: int64

samples  Duration(min)
Subject Task                                 
0       AmusementClip     9216       3.000000
        Baseline         12288       4.000000
        EmoReset          9216       3.000000
        FormL            15360       5.000000
        FormM            18432       6.000000
...                        ...            ...
20      Baseline          9574       3.116536
        EmoReset          9522       3.099609
        FormL            12389       4.032878
        FormM            20274       6.599609
        StressClip        5682       1.849609

[126 rows x 2 columns]

In [ ]:
features = extract_features(
    X,
    column_id="Task",
    column_sort="Timestamp",
    impute_function=impute,
    default_fc_parameters=EfficientFCParameters()
)

In [ ]:
# features.to_csv(EXTRACTED_FEATURES_FILENAME, index=False)
# features = pd.read_csv(sel_features_filename, index_col=0)
display(features)


In [ ]:
sel_features_filename = "../Features/tfresh_eda_cga_features.efficient.selected.csv"
# X_selected.to_csv(sel_features_filename)
X_selected = pd.read_csv(sel_features_filename, index_col=0)
display(X_selected)

In [ ]:
# RFECV

from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.feature_selection import RFECV
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold

RAN_STATE = 21
SPLITS = 10

estimator = RandomForestClassifier(max_depth=3, random_state=RAN_STATE)
cv = StratifiedKFold(n_splits=SPLITS, shuffle=True, random_state=RAN_STATE)
selector = RFECV(estimator=estimator, step=2, cv=cv, scoring="balanced_accuracy")
selector.fit(X_selected, y)
features_mask = selector.support_
X_selected_rfe = X_selected.loc[:, features_mask]

features_scores = { "scores": [], "features": [] }
for score, feat in zip(selector.estimator_.feature_importances_, X_selected_rfe.columns):
    features_scores["scores"].append(score)
    features_scores["features"].append(feat)
features_scores_df = pd.DataFrame(features_scores).sort_values(by="scores", ascending=False)
features_scores_df = features_scores_df.reset_index(drop=True)
features_scores_df.index = features_scores_df.index + 1
print("Selected tsfresh features scores for StressID")
display(features_scores_df)

In [ ]:
# Divergence analysis on StressID
import plotly.express as px

perplexity = np.arange(15, 210, 15)
divergence = []
si_Ncomp = 3

for i in perplexity:
    model = TSNE(n_components=si_Ncomp, init="pca", perplexity=i)
    reduced = model.fit_transform(X_selected)
    divergence.append(model.kl_divergence_)

fig = px.line(x=perplexity, y=divergence, markers=True, width=800, height=600)
fig.update_layout(xaxis_title="Perplexity Values", yaxis_title="KL Divergence")
fig.update_traces(line_color="red", line_width=1)
fig.show()

In [ ]:
# t-SNE in StressID
# Best N-comp=2, Perp=190 np.arange(10, 200, 10)
si_Ncomp = 2
si_Perp = 190

si_tsne = TSNE(n_components=si_Ncomp, perplexity=si_Perp, random_state=RAN_STATE)
si_X_tsne = si_tsne.fit_transform(X_selected)
display(si_tsne.kl_divergence_)

fig = px.scatter(x=si_X_tsne[:,0], y=si_X_tsne[:,1], color=y, width=800, height=600)
fig.update_layout(
    title="t-SNE visualization of StressID dataset",
    xaxis_title="1st t-SNE",
    yaxis_title="2nd t-SNE",
)
fig.show()

In [ ]:
# t-SNE in StressID
# Best N-comp=3, Perp=90 np.arange(15, 210, 15)
si_Ncomp = 3
si_Perp = 90

si_tsne = TSNE(n_components=si_Ncomp, perplexity=si_Perp, random_state=RAN_STATE)
si_X_tsne = si_tsne.fit_transform(X_selected)
display(si_tsne.kl_divergence_)

fig = px.scatter_3d(x=si_X_tsne[:, 0], y=si_X_tsne[:, 1], z=si_X_tsne[:,2], color=y, opacity=0.7, width=800, height=600)
fig.update_layout(title="t-SNE visualization of StressID dataset")
fig.show()